# CO-SPY Fusion v3 — Full PAXL-2-512 Evaluation

Eval **CO-SPY Fusion v3 = Semantic + Artifact + NPR → Fusion v3**
trên generator:

`PixArt-alpha@PixArt-XL-2-512x512`

Protocol của notebook này:

- **Real class:** dùng chung 2.000 ảnh thật từ `Co-Spy_v1/mscoco2017/.../train2017`
- **Fake class:** 2.000 ảnh cho mỗi source PAXL:
  - `mscoco`
  - `flickr`
  - `cc3m`
  - `textcaps`
  - `sbu`
- Mỗi source: **2.000 Real + 2.000 Fake = 4.000 samples**
- 5 source: **20.000 evaluation samples**

> `Co-Spy_v1/stable-diffusion-v1-4/...` **không được dùng** trong eval này vì đó là ảnh AI-generated của SD v1.4, không phải ảnh thật.

Notebook giữ pipeline Fusion v3 đã dùng ở SDXL:
- load raw full Fusion v3 `state_dict`
- bootstrap Semantic / Artifact / NPR từ chính checkpoint tổng hợp
- `strict=True` load toàn bộ checkpoint
- patch artifact features `(B,V,F)`
- preflight 1 batch
- tính AP, AP 95% CI, Accuracy, Real_correct, Fake_detected
- xuất `result.json` và `predictions.csv`


In [1]:
from pathlib import Path
import os, sys, subprocess, time, json, csv, gc

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
REPO = WORK / "cospy"

BATCH = 16
NUM_PER_CLASS = 2000
JPEG_QUALITY = 95
ARTIFACT_VIEWS = 1

SOURCES = ["mscoco", "flickr", "cc3m", "textcaps", "sbu"]
MODEL_NAME = "PixArt-alpha@PixArt-XL-2-512x512"

PAXL_ROOT = Path(
    "/kaggle/input/datasets/datnguyentrung/paxl-2-512"
)

REAL_DIR = Path(
    "/kaggle/input/datasets/datnguyentrung/"
    "co-spy-v1/mscoco2017/mscoco2017/train2017"
)

FUSION_CKPT = Path(
    "/kaggle/input/datasets/datnguyentrung/"
    "co-spy-fusion-v3/checkpoint_best.pth"
)

OUT_DIR = WORK / "eval_paxl_2_512_fusionv3"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("INPUT      :", INPUT)
print("Generator  :", MODEL_NAME)
print("PAXL root  :", PAXL_ROOT)
print("Real dir   :", REAL_DIR)
print("Checkpoint :", FUSION_CKPT)
print("Sources    :", SOURCES)
print("N/class    :", NUM_PER_CLASS)
print("Total eval samples:", NUM_PER_CLASS * 2 * len(SOURCES))


INPUT      : /kaggle/input
Generator  : PixArt-alpha@PixArt-XL-2-512x512
PAXL root  : /kaggle/input/datasets/datnguyentrung/paxl-2-512
Real dir   : /kaggle/input/datasets/datnguyentrung/co-spy-v1/mscoco2017/mscoco2017/train2017
Checkpoint : /kaggle/input/datasets/datnguyentrung/co-spy-fusion-v3/checkpoint_best.pth
Sources    : ['mscoco', 'flickr', 'cc3m', 'textcaps', 'sbu']
N/class    : 2000
Total eval samples: 20000


In [2]:
# ============================================================
# SETUP REPO + DEPENDENCIES
# ============================================================

if not REPO.exists():
    subprocess.run(
        [
            "git", "clone", "--depth", "1",
            "https://github.com/datnguyentrung/CO-SPY.git",
            str(REPO),
        ],
        check=True,
    )

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "open-clip-torch==2.31.0",
    "diffusers==0.32.2",
    "transformers==4.46.3",
    "accelerate==1.0.1",
    "safetensors==0.5.3",
    "ftfy==6.2.3",
    "timm==1.0.15",
    "loguru==0.7.3",
    "scikit-learn",
], check=True)

os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

commit = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"],
    text=True
).strip()

print("Repo ready:", REPO)
print("Commit    :", commit)


Cloning into '/kaggle/working/cospy'...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
urwid 5.1.3 requires wcwidth!=0.9.0,!=0.9.1,>=0.4, but you have wcwidth 0.2.14 which is incompatible.
litellm 1.109.1 requires tokenizers<1.0,>=0.21.0, but you have tokenizers 0.20.3 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
prettytable 3.18.0 requires wcwidth>=0.3.5, but you have wcwidth 0.2.14 which is incompatible.


Repo ready: /kaggle/working/cospy
Commit    : 7e0c8ed


In [3]:
# ============================================================
# PREPARE PAXL FAKE DATA — 5 SOURCES
# Stop as soon as 2,000 images/source are collected.
# ============================================================

from pathlib import Path

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp"}

def collect_images_fast(folder, limit):
    folder = Path(folder)
    assert folder.is_dir(), f"Missing folder: {folder}"

    out = []
    with os.scandir(folder) as it:
        for entry in it:
            if (
                entry.is_file()
                and Path(entry.name).suffix.lower() in IMAGE_EXTS
            ):
                out.append(Path(entry.path))
                if len(out) >= limit:
                    break

    # Fallback for a nested structure.
    if len(out) < limit:
        out = []
        for current_root, dirs, files in os.walk(folder):
            dirs.sort()
            files.sort()
            for name in files:
                p = Path(current_root) / name
                if p.suffix.lower() in IMAGE_EXTS:
                    out.append(p)
                    if len(out) >= limit:
                        return out
    return out


assert PAXL_ROOT.is_dir(), f"PAXL root not found: {PAXL_ROOT}"

fake_dirs = {}
fake_by_source = {}

print("=" * 100)
print("PAXL-2-512 FAKE DATA CHECK")
print("=" * 100)

for source in SOURCES:
    fake_dir = PAXL_ROOT / source / MODEL_NAME
    imgs = collect_images_fast(fake_dir, NUM_PER_CLASS)

    fake_dirs[source] = fake_dir
    fake_by_source[source] = imgs

    print(
        f"{source:10s} | "
        f"{len(imgs):4d}/{NUM_PER_CLASS} images | "
        f"{fake_dir}"
    )

    assert len(imgs) >= NUM_PER_CLASS, (
        f"{source}: fake={len(imgs)}, cần >= {NUM_PER_CLASS}"
    )

print("=" * 100)
print("[PASS] All 5 PAXL fake sources are ready.")


PAXL-2-512 FAKE DATA CHECK
mscoco     | 2000/2000 images | /kaggle/input/datasets/datnguyentrung/paxl-2-512/mscoco/PixArt-alpha@PixArt-XL-2-512x512
flickr     | 2000/2000 images | /kaggle/input/datasets/datnguyentrung/paxl-2-512/flickr/PixArt-alpha@PixArt-XL-2-512x512
cc3m       | 2000/2000 images | /kaggle/input/datasets/datnguyentrung/paxl-2-512/cc3m/PixArt-alpha@PixArt-XL-2-512x512
textcaps   | 2000/2000 images | /kaggle/input/datasets/datnguyentrung/paxl-2-512/textcaps/PixArt-alpha@PixArt-XL-2-512x512
sbu        | 2000/2000 images | /kaggle/input/datasets/datnguyentrung/paxl-2-512/sbu/PixArt-alpha@PixArt-XL-2-512x512
[PASS] All 5 PAXL fake sources are ready.


In [4]:
# ============================================================
# PREPARE REAL CLASS — ONE SHARED MSCOCO SET
# ============================================================

assert REAL_DIR.is_dir(), f"Real MSCOCO folder not found: {REAL_DIR}"

real_images = collect_images_fast(REAL_DIR, NUM_PER_CLASS)

assert len(real_images) >= NUM_PER_CLASS, (
    f"MSCOCO real chỉ có {len(real_images)} ảnh, "
    f"cần {NUM_PER_CLASS}"
)

print("Real class:")
print(" ", REAL_DIR)
print("Found:", len(real_images))
print("First:", real_images[0])

# Explicit guard: SD-v1.4 fake images are NOT used as real.
SD14_FAKE_DIR = Path(
    "/kaggle/input/datasets/datnguyentrung/"
    "co-spy-v1/stable-diffusion-v1-4/stable-diffusion-v1-4"
)

print("\nIgnored SD-v1.4 fake folder:")
print(" ", SD14_FAKE_DIR)

print("\n[PASS] Shared MSCOCO real set ready.")


Real class:
  /kaggle/input/datasets/datnguyentrung/co-spy-v1/mscoco2017/mscoco2017/train2017
Found: 2000
First: /kaggle/input/datasets/datnguyentrung/co-spy-v1/mscoco2017/mscoco2017/train2017/000000000000.jpg

Ignored SD-v1.4 fake folder:
  /kaggle/input/datasets/datnguyentrung/co-spy-v1/stable-diffusion-v1-4/stable-diffusion-v1-4

[PASS] Shared MSCOCO real set ready.


In [5]:
# ============================================================
# VERIFY FULL FUSION V3 CHECKPOINT
# ============================================================

assert FUSION_CKPT.is_file(), (
    f"Không tìm thấy checkpoint: {FUSION_CKPT}"
)

print("Full Fusion V3 checkpoint:")
print(" ", FUSION_CKPT)
print(" Size:", f"{FUSION_CKPT.stat().st_size / 1024**3:.3f} GB")
print("[PASS] Full Fusion V3 checkpoint resolved.")


Full Fusion V3 checkpoint:
  /kaggle/input/datasets/datnguyentrung/co-spy-fusion-v3/checkpoint_best.pth
 Size: 3.593 GB
[PASS] Full Fusion V3 checkpoint resolved.


In [6]:
# ============================================================
# INPUT PREFLIGHT SUMMARY
# ============================================================

import torch

assert torch.cuda.is_available(), (
    "CUDA chưa bật. Kaggle Settings -> Accelerator -> GPU."
)

print("=" * 100)
print("PAXL-2-512 EVAL PREFLIGHT")
print("=" * 100)
print("GPU:", torch.cuda.get_device_name(0))
print(f"✅ real MSCOCO shared: {len(real_images):4d} | {REAL_DIR}")

for source in SOURCES:
    print(
        f"✅ fake {source:10s}: "
        f"{len(fake_by_source[source]):4d} | "
        f"{fake_dirs[source]}"
    )

print(f"✅ full Fusion V3 : {FUSION_CKPT}")
print("=" * 100)
print("[PASS] Dataset + real data + full checkpoint are all ready.")


PAXL-2-512 EVAL PREFLIGHT
GPU: Tesla T4
✅ real MSCOCO shared: 2000 | /kaggle/input/datasets/datnguyentrung/co-spy-v1/mscoco2017/mscoco2017/train2017
✅ fake mscoco    : 2000 | /kaggle/input/datasets/datnguyentrung/paxl-2-512/mscoco/PixArt-alpha@PixArt-XL-2-512x512
✅ fake flickr    : 2000 | /kaggle/input/datasets/datnguyentrung/paxl-2-512/flickr/PixArt-alpha@PixArt-XL-2-512x512
✅ fake cc3m      : 2000 | /kaggle/input/datasets/datnguyentrung/paxl-2-512/cc3m/PixArt-alpha@PixArt-XL-2-512x512
✅ fake textcaps  : 2000 | /kaggle/input/datasets/datnguyentrung/paxl-2-512/textcaps/PixArt-alpha@PixArt-XL-2-512x512
✅ fake sbu       : 2000 | /kaggle/input/datasets/datnguyentrung/paxl-2-512/sbu/PixArt-alpha@PixArt-XL-2-512x512
✅ full Fusion V3 : /kaggle/input/datasets/datnguyentrung/co-spy-fusion-v3/checkpoint_best.pth
[PASS] Dataset + real data + full checkpoint are all ready.


In [7]:
# ============================================================
# EVAL HELPERS — FUSION V3
# Same 2,000 MSCOCO real images are reused for all 5 sources.
# ============================================================

import numpy as np
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import average_precision_score, accuracy_score
from tqdm.auto import tqdm

from utils import png_to_jpeg, bootstrap_ap_ci

device = "cuda:0" if torch.cuda.is_available() else "cpu"
assert device == "cuda:0", "Hãy bật GPU Accelerator trên Kaggle."
print("GPU:", torch.cuda.get_device_name(0))


class SourceImageDataset(Dataset):
    def __init__(
        self,
        source,
        num_per_class,
        sem_transform,
        artifact_transform,
    ):
        self.source = source
        self.n = num_per_class
        self.sem_transform = sem_transform
        self.artifact_transform = artifact_transform

        self.real = real_images[:self.n]
        self.fake = fake_by_source[source][:self.n]

        assert len(self.real) == self.n
        assert len(self.fake) == self.n

        self.real_refs = [str(p) for p in self.real]
        self.fake_refs = [str(p) for p in self.fake]
        self.refs = self.real_refs + self.fake_refs

    def __len__(self):
        return self.n * 2

    def __getitem__(self, idx):
        if idx < self.n:
            image_path = self.real[idx]
            label = 0
        else:
            image_path = self.fake[idx - self.n]
            label = 1

        with Image.open(image_path) as im:
            image = im.convert("RGB")

        # Keep the same preprocessing protocol used by the SDXL eval notebook.
        image = png_to_jpeg(image, quality=JPEG_QUALITY)

        sem_image = self.sem_transform(image)
        artifact_views = self.artifact_transform(image)

        return sem_image, artifact_views, label


def metrics_from_scores(scores, y_true):
    scores = np.asarray(scores, dtype=np.float64)
    y_true = np.asarray(y_true, dtype=np.int64)
    y_pred = (scores >= 0.5).astype(np.int64)

    return {
        "size": int(len(y_true)),
        "num_real": int((y_true == 0).sum()),
        "num_fake": int((y_true == 1).sum()),
        "AP": float(average_precision_score(y_true, scores)),
        "AP_95CI": bootstrap_ap_ci(
            scores.tolist(),
            y_true.tolist(),
            n_samples=500,
            seed=1024,
        ),
        "Accuracy": float(accuracy_score(y_true, y_pred)),
        "Real_correct": float((scores[y_true == 0] < 0.5).mean()),
        "Fake_detected": float((scores[y_true == 1] >= 0.5).mean()),
    }


def eval_source_v3(model, source):
    ds = SourceImageDataset(
        source=source,
        num_per_class=NUM_PER_CLASS,
        sem_transform=model.sem.test_transform,
        artifact_transform=model.artifact_view_transform(
            ARTIFACT_VIEWS
        ),
    )

    loader = DataLoader(
        ds,
        batch_size=BATCH,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
    )

    scores, labels_all = [], []
    t0 = time.time()

    with torch.inference_mode():
        for sem_images, artifact_views, labels in tqdm(
            loader,
            desc=f"Fusion v3 | {source}",
            unit="batch",
        ):
            assert artifact_views.ndim == 5, (
                "Expected raw artifact views (B,V,C,H,W), got "
                f"{tuple(artifact_views.shape)}"
            )

            pred = model.predict_views(
                sem_images,
                artifact_views,
            )

            scores.extend(float(v) for v in pred)
            labels_all.extend(labels.tolist())

    result = metrics_from_scores(scores, labels_all)
    result["Runtime_minutes"] = float((time.time() - t0) / 60)

    return (
        result,
        np.asarray(scores),
        np.asarray(labels_all),
        ds.refs,
    )


print("[PASS] Fusion v3 eval helpers ready.")


GPU: Tesla T4
[PASS] Fusion v3 eval helpers ready.


In [8]:
# ============================================================
# LOAD FULL FUSION V3 CHECKPOINT
#
# The checkpoint contains:
#   sem.* + art.* + npr.* + fusion_head.*
#
# CoSpyFusionDetector's constructor expects three branch
# checkpoint files, so TEMP bootstrap branch checkpoints are
# extracted from this same full state_dict. Then the complete
# full state_dict is loaded again with strict=True.
# ============================================================

from detectors.sd_v1_4 import CoSpyFusionDetector

fusion_path = FUSION_CKPT
assert fusion_path.is_file(), f"Missing checkpoint: {fusion_path}"

print("Loading full Fusion V3 checkpoint...")
print("Fusion v3:", fusion_path)

state = torch.load(
    fusion_path,
    map_location="cpu",
    weights_only=True,
)

assert isinstance(state, dict), type(state)

required_prefixes = [
    "sem.",
    "art.",
    "npr.",
    "fusion_head.",
]

for prefix in required_prefixes:
    assert any(k.startswith(prefix) for k in state), (
        f"Fusion checkpoint does not contain required prefix: {prefix}"
    )

print("state_dict keys:", len(state))
print("Has sem.*        :", any(k.startswith("sem.") for k in state))
print("Has art.*        :", any(k.startswith("art.") for k in state))
print("Has npr.*        :", any(k.startswith("npr.") for k in state))
print("Has fusion_head.*:", any(k.startswith("fusion_head.") for k in state))

BOOTSTRAP_DIR = WORK / "fusion_v3_bootstrap"
BOOTSTRAP_DIR.mkdir(parents=True, exist_ok=True)


def save_submodule_state(full_state, prefix, out_path):
    sub_state = {
        k[len(prefix):]: v
        for k, v in full_state.items()
        if k.startswith(prefix)
    }
    assert sub_state, f"No keys found for prefix: {prefix}"
    torch.save(sub_state, out_path)
    return out_path


semantic_path = save_submodule_state(
    state, "sem.", BOOTSTRAP_DIR / "semantic.pth"
)
artifact_path = save_submodule_state(
    state, "art.", BOOTSTRAP_DIR / "artifact.pth"
)
npr_path = save_submodule_state(
    state, "npr.", BOOTSTRAP_DIR / "npr.pth"
)

print("\nBuilding CO-SPY Fusion v3 runtime...")
print("Artifact bootstrap:", artifact_path)
print("Semantic bootstrap:", semantic_path)
print("NPR bootstrap     :", npr_path)

model = CoSpyFusionDetector(
    semantic_weights_path=str(semantic_path),
    artifact_weights_path=str(artifact_path),
    npr_weights_path=str(npr_path),
    fusion_version="v3",
    branch_dropout=0.15,
    artifact_views=ARTIFACT_VIEWS,
)

incompat = model.load_state_dict(state, strict=False)

print("\nCompatibility report")
print("Missing keys   :", len(incompat.missing_keys))
print("Unexpected keys:", len(incompat.unexpected_keys))

if incompat.missing_keys:
    print("\nFirst missing keys:")
    for k in incompat.missing_keys[:30]:
        print(" ", k)

if incompat.unexpected_keys:
    print("\nFirst unexpected keys:")
    for k in incompat.unexpected_keys[:30]:
        print(" ", k)

assert not incompat.missing_keys, (
    "Checkpoint is missing model keys; stop before evaluation."
)
assert not incompat.unexpected_keys, (
    "Checkpoint contains unexpected model keys; stop before evaluation."
)

model.load_state_dict(state, strict=True)

del state
gc.collect()

model.to(device)
model.eval()

print("\n[PASS] Full Fusion V3 checkpoint loaded successfully.")
print("Device:", device)
print("GPU   :", torch.cuda.get_device_name(0))


The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

I0000 00:00:1795193839.075611      24 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


Loading full Fusion V3 checkpoint...
Fusion v3: /kaggle/input/datasets/datnguyentrung/co-spy-fusion-v3/checkpoint_best.pth
state_dict keys: 1232
Has sem.*        : True
Has art.*        : True
Has npr.*        : True
Has fusion_head.*: True

Building CO-SPY Fusion v3 runtime...
Artifact bootstrap: /kaggle/working/fusion_v3_bootstrap/artifact.pth
Semantic bootstrap: /kaggle/working/fusion_v3_bootstrap/semantic.pth
NPR bootstrap     : /kaggle/working/fusion_v3_bootstrap/npr.pth


open_clip_model.safetensors:   0%|          | 0.00/3.51G [00:00<?, ?B/s]

config.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

vae/diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]


Compatibility report
Missing keys   : 0
Unexpected keys: 0

[PASS] Full Fusion V3 checkpoint loaded successfully.
Device: cuda:0
GPU   : Tesla T4


In [9]:
# ============================================================
# PATCH FUSION V3 EVAL RUNTIME
# Accept artifact feature views shaped (B, V, F)
# ============================================================

original_prepare_artifact_views = model._prepare_artifact_views

def patched_prepare_artifact_views(x_art):
    # Raw artifact views enter as (B,V,C,H,W).
    # extract_features() turns them into (B,V,F).
    if x_art.ndim == 3:
        return x_art

    return original_prepare_artifact_views(x_art)

model._prepare_artifact_views = patched_prepare_artifact_views

print("[PATCH] Fusion v3 now accepts artifact feature views (B, V, F).")


[PATCH] Fusion v3 now accepts artifact feature views (B, V, F).


In [10]:
# ============================================================
# PREFLIGHT PAXL — TEST EXACTLY 1 BATCH BEFORE FULL EVAL
# ============================================================

test_source = "mscoco"

test_ds = SourceImageDataset(
    source=test_source,
    num_per_class=min(NUM_PER_CLASS, 32),
    sem_transform=model.sem.test_transform,
    artifact_transform=model.artifact_view_transform(
        ARTIFACT_VIEWS
    ),
)

test_loader = DataLoader(
    test_ds,
    batch_size=min(BATCH, 16),
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

sem_images, artifact_views, labels = next(iter(test_loader))

print("Before normalization:")
print("  sem_images     :", tuple(sem_images.shape))
print("  artifact_views :", tuple(artifact_views.shape))
print("  labels         :", tuple(labels.shape))

assert artifact_views.ndim == 5, (
    f"Expected raw artifact views (B,V,C,H,W), got "
    f"{tuple(artifact_views.shape)}"
)

print("\nTesting end-to-end forward...")

with torch.inference_mode():
    probe_pred = model.predict_views(
        sem_images,
        artifact_views,
    )

print("\nPrediction probe:")
print("  count :", len(probe_pred))
print("  first :", probe_pred[:5])

assert len(probe_pred) == len(labels)
assert all(0.0 <= float(x) <= 1.0 for x in probe_pred)

print("\n[PASS] One-batch PAXL Fusion v3 inference works.")


Before normalization:
  sem_images     : (16, 3, 384, 384)
  artifact_views : (16, 1, 3, 224, 224)
  labels         : (16,)

Testing end-to-end forward...

Prediction probe:
  count : 16
  first : [1.8218381664381056e-11, 1.3540606857986859e-08, 4.0974279633294863e-13, 2.3178692742931162e-08, 1.2079116873398021e-10]

[PASS] One-batch PAXL Fusion v3 inference works.


In [11]:
# ============================================================
# RUN FULL PAXL-2-512 — 5 SOURCES
# ============================================================

results = {}
outputs = {}
all_scores = []
all_labels = []

total_start = time.time()

for source in SOURCES:
    r, scores, labels, refs = eval_source_v3(
        model,
        source,
    )

    results[source] = r
    outputs[source] = {
        "scores": scores,
        "labels": labels,
        "refs": refs,
    }

    all_scores.append(scores)
    all_labels.append(labels)

    print(
        f"{source:8s} | "
        f"AP={r['AP']*100:6.2f}% | "
        f"Acc={r['Accuracy']*100:6.2f}% | "
        f"Real={r['Real_correct']*100:6.2f}% | "
        f"Fake={r['Fake_detected']*100:6.2f}% | "
        f"{r['Runtime_minutes']:.1f} min"
    )

all_scores = np.concatenate(all_scores)
all_labels = np.concatenate(all_labels)
all_pred = (all_scores >= 0.5).astype(int)

macro_ap = float(
    np.mean([results[s]["AP"] for s in SOURCES])
)
macro_acc = float(
    np.mean([results[s]["Accuracy"] for s in SOURCES])
)

pooled_ap = float(
    average_precision_score(all_labels, all_scores)
)
pooled_acc = float(
    accuracy_score(all_labels, all_pred)
)

summary = {
    "benchmark": "Co-Spy-Bench / PAXL-2-512",
    "generator": MODEL_NAME,
    "model": "CO-SPY Fusion v3",
    "architecture": "Semantic + Artifact + NPR -> Fusion v3",
    "real_protocol": "Same 2000 MSCOCO train2017 real images reused for every source",
    "num_per_class_per_source": NUM_PER_CLASS,
    "jpeg_quality": JPEG_QUALITY,
    "artifact_views": ARTIFACT_VIEWS,
    "sources": SOURCES,
    "per_source": results,
    "Macro_AP": macro_ap,
    "Macro_Accuracy": macro_acc,
    "Pooled_AP": pooled_ap,
    "Pooled_Accuracy": pooled_acc,
    "Total_runtime_minutes": float(
        (time.time() - total_start) / 60
    ),
}

print("\n" + "=" * 78)
print("PAXL-2-512 / CO-SPY FUSION V3")
print("=" * 78)

for source in SOURCES:
    ci = results[source]["AP_95CI"]
    print(
        f"{source:8s} | "
        f"AP {results[source]['AP']*100:6.2f}% "
        f"[{ci[0]*100:.2f}, {ci[1]*100:.2f}] | "
        f"Acc {results[source]['Accuracy']*100:6.2f}% | "
        f"Real {results[source]['Real_correct']*100:6.2f}% | "
        f"Fake {results[source]['Fake_detected']*100:6.2f}%"
    )

print("-" * 78)
print(f"Macro AP       : {macro_ap*100:.2f}%")
print(f"Macro Accuracy : {macro_acc*100:.2f}%")
print(f"Pooled AP      : {pooled_ap*100:.2f}%")
print(f"Pooled Accuracy: {pooled_acc*100:.2f}%")
print(
    f"Total runtime  : "
    f"{summary['Total_runtime_minutes']:.1f} min"
)
print("=" * 78)


Fusion v3 | mscoco:   0%|          | 0/250 [00:00<?, ?batch/s]

mscoco   | AP= 99.83% | Acc= 85.78% | Real=100.00% | Fake= 71.55% | 36.8 min


Fusion v3 | flickr:   0%|          | 0/250 [00:00<?, ?batch/s]

flickr   | AP= 99.87% | Acc= 87.60% | Real=100.00% | Fake= 75.20% | 34.1 min


Fusion v3 | cc3m:   0%|          | 0/250 [00:00<?, ?batch/s]

cc3m     | AP= 99.77% | Acc= 82.20% | Real=100.00% | Fake= 64.40% | 34.6 min


Fusion v3 | textcaps:   0%|          | 0/250 [00:00<?, ?batch/s]

textcaps | AP= 99.83% | Acc= 84.62% | Real=100.00% | Fake= 69.25% | 34.7 min


Fusion v3 | sbu:   0%|          | 0/250 [00:00<?, ?batch/s]

sbu      | AP= 99.63% | Acc= 78.57% | Real=100.00% | Fake= 57.15% | 34.7 min

PAXL-2-512 / CO-SPY FUSION V3
mscoco   | AP  99.83% [99.76, 99.88] | Acc  85.78% | Real 100.00% | Fake  71.55%
flickr   | AP  99.87% [99.82, 99.92] | Acc  87.60% | Real 100.00% | Fake  75.20%
cc3m     | AP  99.77% [99.68, 99.84] | Acc  82.20% | Real 100.00% | Fake  64.40%
textcaps | AP  99.83% [99.76, 99.88] | Acc  84.62% | Real 100.00% | Fake  69.25%
sbu      | AP  99.63% [99.51, 99.73] | Acc  78.57% | Real 100.00% | Fake  57.15%
------------------------------------------------------------------------------
Macro AP       : 99.79%
Macro Accuracy : 83.75%
Pooled AP      : 99.79%
Pooled Accuracy: 83.75%
Total runtime  : 175.0 min


In [12]:
# ============================================================
# SAVE RESULTS
# ============================================================

result_path = OUT_DIR / "result.json"
pred_path = OUT_DIR / "predictions.csv"

with open(result_path, "w", encoding="utf-8") as f:
    json.dump(
        summary,
        f,
        indent=2,
        ensure_ascii=False,
    )

with open(
    pred_path,
    "w",
    newline="",
    encoding="utf-8",
) as f:
    w = csv.writer(f)
    w.writerow([
        "source",
        "path_or_ref",
        "label",
        "fusion_v3_score",
        "predicted_label",
    ])

    for source in SOURCES:
        o = outputs[source]

        for ref, y, score in zip(
            o["refs"],
            o["labels"],
            o["scores"],
        ):
            w.writerow([
                source,
                ref,
                int(y),
                float(score),
                int(score >= 0.5),
            ])

print("PAXL-2-512 evaluation saved:")
print(" ", result_path)
print(" ", pred_path)

print("\nRESULT JSON:")
print(json.dumps(summary, indent=2, ensure_ascii=False))


PAXL-2-512 evaluation saved:
  /kaggle/working/eval_paxl_2_512_fusionv3/result.json
  /kaggle/working/eval_paxl_2_512_fusionv3/predictions.csv

RESULT JSON:
{
  "benchmark": "Co-Spy-Bench / PAXL-2-512",
  "generator": "PixArt-alpha@PixArt-XL-2-512x512",
  "model": "CO-SPY Fusion v3",
  "architecture": "Semantic + Artifact + NPR -> Fusion v3",
  "real_protocol": "Same 2000 MSCOCO train2017 real images reused for every source",
  "num_per_class_per_source": 2000,
  "jpeg_quality": 95,
  "artifact_views": 1,
  "sources": [
    "mscoco",
    "flickr",
    "cc3m",
    "textcaps",
    "sbu"
  ],
  "per_source": {
    "mscoco": {
      "size": 4000,
      "num_real": 2000,
      "num_fake": 2000,
      "AP": 0.9983048723574423,
      "AP_95CI": [
        0.9976206268984008,
        0.9988449233115304
      ],
      "Accuracy": 0.85775,
      "Real_correct": 1.0,
      "Fake_detected": 0.7155,
      "Runtime_minutes": 36.791188840071364
    },
    "flickr": {
      "size": 4000,
      "num_rea